# klygo.archive (io.py): Create and extract archives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Io.ipynb)


This notebook favors runnable workflows over long reference tables. It uses only generated data inside a resettable temporary directory.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample data

In [1]:
from tempfile import TemporaryDirectory
from zipfile import BadZipFile
from klygo import archive, files

if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-archive-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-archive-_uf1gv83


In [2]:
source = files.mkdir(root / "dataset")
files.save(source / "labels.txt", ["cat", "dog"], verbose=False)
files.save(source / "config.json", {"batch": 16, "classes": 2}, verbose=False)
images = files.mkdir(source / "images")
files.save(images / "cat.txt", "image placeholder", verbose=False)

sample_archive = root / "dataset.zip"
archive.compress(source, sample_archive, verbose=False)
print("Members:", archive.list_files(sample_archive))

Members: ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


## Jump to an API

[`compress`](#compress) · [`extract`](#extract) · [`extract_file`](#extract_file) · [`extract_matching`](#extract_matching)

<a id="compress"></a>

## `archive.compress`

Create an archive from a file or directory.

**Signature:** `archive.compress(source: Union[str, pathlib.Path], output_path: Union[str, pathlib.Path], format: Optional[str] = None, overwrite: bool = False, verbose: bool = True, compresslevel: int = 6, method: Optional[str] = None, follow_symlinks: bool = False, include_root: bool = True) -> None`

Creates an archive as a side effect. The destination format comes from `format` or the output extension. Existing destinations require `overwrite=True`.

Each block below is runnable in order.

**Basic use**

In [3]:
zip_path = root / "training.zip"
archive.compress(source, zip_path, verbose=False)
print(zip_path, archive.detect_format(zip_path))
print(archive.list_files(zip_path))

C:\Users\Mine\AppData\Local\Temp\klygo-archive-_uf1gv83\training.zip zip
['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


**Practical variation**

In [4]:
contents_only = root / "contents-only.tar.gz"
archive.compress(
    source,
    contents_only,
    include_root=False,
    compresslevel=9,
    verbose=False,
)
print(archive.list_files(contents_only))

['config.json', 'images/cat.txt', 'labels.txt']


**Expected error or edge behavior**

In [5]:
try:
    archive.compress(source, sample_archive, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

archive.compress(source, sample_archive, overwrite=True, verbose=False)
print("Rebuilt intentionally:", sample_archive)

FileExistsError: output_path already exists: C:\Users\Mine\AppData\Local\Temp\klygo-archive-_uf1gv83\dataset.zip
Rebuilt intentionally: C:\Users\Mine\AppData\Local\Temp\klygo-archive-_uf1gv83\dataset.zip


<a id="extract"></a>

## `archive.extract`

Extract selected or all members from an archive.

**Signature:** `archive.extract(archive_path: Union[str, pathlib.Path], output_dir: Union[str, pathlib.Path] = '.', overwrite: bool = False, verbose: bool = True, password: Optional[str] = None, include: Union[str, List[str], NoneType] = None, exclude: Union[str, List[str], NoneType] = None) -> None`

Extracts selected members when `include` or `exclude` is supplied. Backends reject unsafe member paths rather than writing outside the destination.

Each block below is runnable in order.

**Basic use**

In [6]:
all_files = root / "all-files"
archive.extract(sample_archive, all_files, verbose=False)
print([files.as_posix(files.relative(path, all_files)) for path in files.find(all_files)])

['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


**Practical variation**

In [7]:
json_only = root / "json-only"
archive.extract(sample_archive, json_only, include="*.json", verbose=False)
print([files.name(path) for path in files.find(json_only)])

['config.json']


**Expected error or edge behavior**

In [8]:
try:
    archive.extract(sample_archive, all_files, verbose=False)
except FileExistsError as error:
    print("FileExistsError:", error)

archive.extract(sample_archive, all_files, overwrite=True, verbose=False)

FileExistsError: Files already exist in output directory: dataset/config.json, dataset/images/cat.txt, dataset/labels.txt. Use overwrite=True.


<a id="extract_file"></a>

## `archive.extract_file`

Extract one archive member using streaming I/O.

**Signature:** `archive.extract_file(archive_path: Union[str, pathlib.Path], filename: str, output_dir: Union[str, pathlib.Path] = '.', overwrite: bool = False, password: Optional[str] = None) -> None`

Extracts one exact stored member name. Obtain the canonical name with `list_files()` when directory prefixes are uncertain.

Each block below is runnable in order.

**Basic use**

In [9]:
label_member = next(name for name in archive.list_files(sample_archive) if name.endswith("labels.txt"))
single_output = root / "single"
archive.extract_file(sample_archive, label_member, single_output)
print(files.load(single_output / files.name(label_member), as_lines=True, verbose=False))

['cat', 'dog']


**Practical variation**

In [10]:
config_member = next(name for name in archive.list_files(sample_archive) if name.endswith("config.json"))
archive.extract_file(sample_archive, config_member, root / "config-output")
print(files.load(root / "config-output" / files.name(config_member), verbose=False))

{'batch': 16, 'classes': 2}


**Expected error or edge behavior**

In [11]:
try:
    archive.extract_file(sample_archive, "missing.txt", root / "missing-member")
except KeyError as error:
    print("KeyError:", error)

KeyError: "'missing.txt' not found in archive. Use list_files() to see available file_utils."


<a id="extract_matching"></a>

## `archive.extract_matching`

Extract members matching a wildcard pattern.

**Signature:** `archive.extract_matching(archive_path: Union[str, pathlib.Path], pattern: str, output_dir: Union[str, pathlib.Path] = '.', overwrite: bool = False, password: Optional[str] = None) -> None`

Uses backend glob matching and preserves stored directory structure under the output directory.

Each block below is runnable in order.

**Basic use**

In [12]:
text_output = root / "text-members"
archive.extract_matching(sample_archive, "*.txt", text_output)
print([files.as_posix(files.relative(path, text_output)) for path in files.find(text_output)])

['dataset/images/cat.txt', 'dataset/labels.txt']


**Practical variation**

In [13]:
json_output = root / "json-members"
archive.extract_matching(sample_archive, "*.json", json_output)
print([files.name(path) for path in files.find(json_output)])

['config.json']


**Expected error or edge behavior**

In [14]:
empty_output = root / "no-matches"
archive.extract_matching(sample_archive, "*.csv", empty_output)
print("Matches:", files.find(empty_output) if files.exists(empty_output) else [])

Matches: []


## Put it together

In [15]:
bundle_archive = root / "release_bundle.tar.gz"
archive.compress(source, bundle_archive, include_root=False, verbose=False)
restored_dir = root / "restored_json"
archive.extract_matching(bundle_archive, "*.json", restored_dir)
print("Restored files:", [files.name(p) for p in files.find(restored_dir)])

Restored files: ['config.json']


## Remember

- `compress()` and `extract()` protect existing files by default; pass `overwrite=True` only when replacing an existing archive or output file.
- Use `extract_file()` for a single known member path and `extract_matching()` for glob-based partial extraction.
- Next in `klygo.archive`: continue with [Document_Archive_Metadata.ipynb](Document_Archive_Metadata.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Metadata.ipynb)) or return to [Document_Archive_Overview.ipynb](Document_Archive_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Overview.ipynb)).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).